# Niveau 3 — Nature des candidats plausibles

Classification visuelle des supports plausibles.


In [1]:

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output
from PIL import Image


ROOT = Path.cwd()

CSV_PATH = (
    ROOT
    / "data"
    / "niveau3_multizone"
    / "candidats"
    / "candidats_uniques.csv"
)

ZONES_DIR = (
    ROOT
    / "data"
    / "niveau3_multizone"
    / "zones"
)


df = pd.read_csv(CSV_PATH)

for col in [
    "review_status",
    "review_comment",
    "support_type",
]:
    if col not in df.columns:
        df[col] = ""

    df[col] = (
        df[col]
        .fillna("")
        .astype("object")
    )


review_indices = df[
    df["review_status"]
    .astype(str)
    .eq("plausible")
].index.tolist()


print("Candidats plausibles :", len(review_indices))


current_position = 0

output = widgets.Output()


buttons_data = [
    ("🏗️ Pylône", "pylone", "success"),
    ("📡 Mât", "mat", "success"),
    ("🏢 Bâtiment", "batiment", ""),
    ("💧 Château d'eau", "chateau_eau", ""),
    ("🌾 Silo", "silo", ""),
    ("🔧 Autre support", "autre_support", ""),
    ("❓ Indéterminé", "indetermine", "warning"),
]


buttons = []


def save():
    df.to_csv(
        CSV_PATH,
        index=False,
    )


def show_candidate():

    global current_position

    with output:

        clear_output(wait=True)

        idx = review_indices[current_position]
        row = df.loc[idx]

        uid = str(
            row["unique_candidate_id"]
        )

        zone_id = str(
            row["zone_id"]
        )

        image_name = str(
            row["image"]
        )

        image_path = (
            ZONES_DIR
            / zone_id
            / "images"
            / image_name
        )


        print(
            f"Candidat "
            f"{current_position + 1}/"
            f"{len(review_indices)}"
        )

        print("ID :", uid)
        print("Zone :", zone_id)

        print(
            "Confiance :",
            f"{float(row['confidence']):.3f}"
        )

        print(
            "Distance ANFR :",
            f"{float(row['nearest_ANFR_distance_m']):.1f} m"
        )

        current_type = str(
            row["support_type"]
        ).strip()

        print(
            "Type actuel :",
            current_type
            if current_type
            else "non classé"
        )


        image = Image.open(
            image_path
        ).convert("RGB")


        x1 = int(float(row["x1"]))
        y1 = int(float(row["y1"]))
        x2 = int(float(row["x2"]))
        y2 = int(float(row["y2"]))


        margin = 180

        cx1 = max(0, x1 - margin)
        cy1 = max(0, y1 - margin)
        cx2 = min(image.width, x2 + margin)
        cy2 = min(image.height, y2 + margin)


        context = image.crop(
            (
                cx1,
                cy1,
                cx2,
                cy2,
            )
        )


        fig, ax = plt.subplots(
            figsize=(7, 7)
        )

        ax.imshow(context)

        from matplotlib.patches import Rectangle

        rect = Rectangle(
            (
                x1 - cx1,
                y1 - cy1,
            ),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=3,
            edgecolor="red",
        )

        ax.add_patch(rect)

        ax.set_title(
            f"{uid} | {zone_id}"
        )

        ax.axis("off")

        plt.show()


def set_type(value):

    global current_position

    idx = review_indices[
        current_position
    ]

    df.at[
        idx,
        "support_type"
    ] = value

    save()


    if (
        current_position
        < len(review_indices) - 1
    ):
        current_position += 1
        show_candidate()

    else:

        with output:

            clear_output(wait=True)

            print(
                "✅ Classification terminée."
            )

            plausible_df = df[
                df["review_status"]
                .astype(str)
                .eq("plausible")
            ]

            print(
                plausible_df[
                    "support_type"
                ]
                .replace(
                    "",
                    "non_classe"
                )
                .value_counts()
            )


for label, value, style in buttons_data:

    btn = widgets.Button(
        description=label,
        button_style=style,
    )

    btn.on_click(
        lambda _,
        value=value:
        set_type(value)
    )

    buttons.append(btn)


display(output)

display(
    widgets.VBox(
        [
            widgets.HBox(
                buttons[:4]
            ),
            widgets.HBox(
                buttons[4:]
            ),
        ]
    )
)

show_candidate()


Candidats plausibles : 27


Output()